*© 2026 Paul Fergus. Free for student and research use — commercial use is strictly prohibited.*

# Lab 13 — Inside CLIP: contrastive learning and multi-modal retrieval

**Module:** Deep Learning Concepts and Techniques (Computer Vision)
**Week:** 13
**Estimated time:** 210 minutes

---

## Learning outcomes

By the end of this lab you should be able to:

1. Explain **contrastive learning** — how CLIP is trained to pull matching (image, text) pairs together and push non-matching pairs apart in a shared embedding space — and compute the contrastive loss from first principles on a toy batch.
2. Compute image and text embeddings with a pretrained CLIP model and compare them with cosine similarity.
3. Build a **zero-shot image classifier** directly from CLIP embeddings, with no training, and explain the mechanism that makes this possible.
4. Build a **text-to-image retrieval** ("image search") system over a real image collection using nothing but embeddings and cosine similarity.
5. Visualise CLIP's embedding space and reason about what structure it captures despite having never been trained on our specific classes.
6. Explain how prompt wording changes CLIP's embeddings, and connect this to Lab 9's zero-shot detection prompt-engineering findings.

## Prerequisites

- **Lab 9** — you already used CLIP, indirectly, as YOLOE's text encoder for zero-shot prompts. Today we open that black box.
- **Lab 12** — self-attention and Transformers; CLIP's image encoder is a ViT.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 9 — multi-modal models.
- Lecture 13: *Contrastive pretraining and the joint embedding space — how CLIP connects pixels and words*.

## The thread from last week

Lab 9 handed you a black box: call `yoloe.set_classes(names, yoloe.get_text_pe(names))`, and text prompts started detecting objects. That worked because YOLOE reuses a **CLIP** text encoder internally — the same `clip-anytorch` package already installed in this container specifically for that lab. You used it without ever calling `import clip` yourself. Today you do.

**CLIP** (Contrastive Language-Image Pre-training, Radford et al., 2021) is trained on 400 million (image, caption) pairs scraped from the web, with one objective: given a batch of images and a batch of captions, correctly match each image to its caption and vice versa. Nothing in that objective mentions "buffalo" or "zebra" or any fixed class list — which is exactly why it generalises to classes it was never explicitly taught, the same property that made Lab 9's zero-shot detection work at all.

## Useful references

- Radford, A. et al. (2021), *Learning Transferable Visual Models From Natural Language Supervision* — the CLIP paper.
- [OpenAI CLIP repository](https://github.com/openai/CLIP) — the original model and usage examples (`clip-anytorch` mirrors this exact API).
- Oord, A. van den, Li, Y. and Vinyals, O. (2018), *Representation Learning with Contrastive Predictive Coding* — the broader contrastive learning framework CLIP builds on.

---

## 1. Contrastive learning, from first principles

CLIP trains **two encoders** — an image encoder and a text encoder — that project their very different inputs into the **same** fixed-size embedding space. Training proceeds in batches of `N` (image, caption) pairs, scraped so that image `i` and caption `i` genuinely describe each other:

1. Encode all `N` images and all `N` captions into embeddings, then **L2-normalise** each one (so every embedding has length 1 — cosine similarity then becomes a plain dot product).
2. Compute the full `N x N` matrix of cosine similarities between every image and every caption.
3. The **diagonal** of that matrix is the "correct" pairs (image `i` with its own caption `i`); every off-diagonal entry is a mismatched pair.
4. Train with a loss that pushes the diagonal entries **up** and the off-diagonal entries **down** — in both directions at once (image-to-text *and* text-to-image), which is why it's called a *symmetric* contrastive loss.

Nothing here is specific to any fixed class list. The model only ever learns "does this image match this text", which is precisely general enough to work on text it has never seen paired with anything during training — including four wildlife species it was never told about by name.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

RNG_SEED = 7144
torch.manual_seed(RNG_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


def clip_contrastive_loss(image_embeds: torch.Tensor, text_embeds: torch.Tensor, temperature: float = 0.07):
    """CLIP's symmetric contrastive loss, from first principles.
    image_embeds, text_embeds: (N, d) — embedding i of each should match embedding i of the other.
    Returns (loss, similarity_matrix).
    """
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)

    logits = (image_embeds @ text_embeds.T) / temperature   # (N, N) cosine similarities, scaled
    labels = torch.arange(logits.shape[0], device=logits.device)  # the diagonal is "correct"

    loss_i2t = F.cross_entropy(logits, labels)       # each image should pick its own caption
    loss_t2i = F.cross_entropy(logits.T, labels)      # each caption should pick its own image
    loss = (loss_i2t + loss_t2i) / 2
    return loss, logits


# Toy batch: 5 "images" and 5 "captions", represented as random embeddings.
# We rig embeddings 0 to be an obviously-matched pair to see the loss respond.
N, d = 5, 16
torch.manual_seed(RNG_SEED)
image_embeds = torch.randn(N, d)
text_embeds = torch.randn(N, d)

loss_random, sim_random = clip_contrastive_loss(image_embeds, text_embeds)
print(f"Loss with random (unaligned) embeddings: {loss_random.item():.4f}")

# Now make every image/text pair i EXACTLY match (as if training had converged).
text_embeds_matched = image_embeds.clone()
loss_matched, sim_matched = clip_contrastive_loss(image_embeds, text_embeds_matched)
print(f"Loss with perfectly matched embeddings:  {loss_matched.item():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(sim_random.detach().numpy(), cmap="RdBu_r"); axes[0].set_title(f"Random pairs (loss={loss_random:.2f})")
axes[1].imshow(sim_matched.detach().numpy(), cmap="RdBu_r"); axes[1].set_title(f"Matched pairs (loss={loss_matched:.2f})")
for ax in axes:
    ax.set_xlabel("caption index"); ax.set_ylabel("image index")
plt.tight_layout(); plt.show()

**Reading the two similarity matrices.** With random embeddings, high similarity scatters unpredictably across the grid — no structure, high loss. With matched embeddings, the diagonal lights up strongly and the loss collapses toward zero, because cross-entropy over each row (and each column) now has almost all its probability mass on the correct index. Training CLIP on 400 million real pairs is the same loss, at scale, pushing the diagonal up and everything else down until the model has learned a genuinely useful shared space — not just for the pairs it saw, but for new images and new text that share the same underlying structure.

## 2. Loading a pretrained CLIP model

`clip-anytorch` — already installed in this container for Lab 9 — provides the exact same `import clip` API as OpenAI's original release. We load `ViT-B/32`: a Vision Transformer image encoder (the same architecture family as Lab 12, just trained completely differently) paired with a Transformer text encoder.

> **First run downloads the checkpoint** (~350 MB). If you see `ModuleNotFoundError: No module named 'clip'`, the container should already have it installed for Lab 9 — if not, `pip install --user clip-anytorch ftfy regex` and restart the kernel, exactly as Lab 9's recovery cell does.

In [ ]:
import clip
from PIL import Image

clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()
n_params = sum(p.numel() for p in clip_model.parameters())
print(f"CLIP ViT-B/32 loaded. Parameters: {n_params:,}")
print(f"Image preprocessing pipeline: {clip_preprocess}")

### A first look — does an image match its caption?

We'll reuse Lab 6/10's wildlife photographs — real images CLIP has certainly never seen during its own training, and whose species names (buffalo, elephant, rhino, zebra) it was never explicitly taught as a fixed class list.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("../lab06_image_annotation/data")
CLASSES = [ln.strip() for ln in (DATA_ROOT / "classes.txt").read_text().splitlines() if ln.strip()]
sample_imgs = sorted((DATA_ROOT / "images" / "test").glob("*.jpg"))[:1]
img_path = sample_imgs[0]

image_input = clip_preprocess(Image.open(img_path)).unsqueeze(0).to(DEVICE)
candidate_texts = [f"a photo of a {c}" for c in CLASSES]
text_tokens = clip.tokenize(candidate_texts).to(DEVICE)

with torch.no_grad():
    logits_per_image, _ = clip_model(image_input, text_tokens)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(Image.open(img_path)); axes[0].axis("off"); axes[0].set_title(img_path.name, fontsize=9)
axes[1].barh(candidate_texts, probs, color="#1c7ed6")
axes[1].set_xlabel("probability"); axes[1].set_xlim(0, 1)
axes[1].set_title(f"CLIP's similarity, softmaxed over {len(CLASSES)} prompts")
plt.tight_layout(); plt.show()

for text, p in zip(candidate_texts, probs):
    print(f"  {text:<30s} {p:.4f}")

That `logits_per_image` call is doing exactly what Section 1's `clip_contrastive_loss` computed by hand: L2-normalise both embeddings, take the dot product, scale by a temperature. `model()` just does it for you and hands back the raw similarity scores (which CLIP calls "logits" since they feed directly into a softmax, just like a classifier's).

## 3. Zero-shot whole-image classification

Section 2 classified one image against four prompts by hand. Let's do it properly, at scale, and check accuracy against real labels — the same rigor Lab 8 demanded of Lab 7's detector, now applied to a model that was never trained on our data at all.

**One wrinkle specific to whole-image classification:** our wildlife photos can contain *multiple* species in one frame (a herd shot might have both zebra and buffalo). "What is the class of this image" is only a well-posed question for images containing exactly one species — so we filter to single-species images before evaluating, and say so explicitly rather than silently biasing the result.

In [ ]:
def read_yolo_labels(label_path: Path):
    if not label_path.is_file():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        boxes.append(int(line.split()[0]))
    return boxes


test_img_dir = DATA_ROOT / "images" / "test"
test_lbl_dir = DATA_ROOT / "labels" / "test"

single_species_images = []
for img_path in sorted(test_img_dir.glob("*.jpg")):
    lbl_path = test_lbl_dir / f"{img_path.stem}.txt"
    classes_present = set(read_yolo_labels(lbl_path))
    if len(classes_present) == 1:
        single_species_images.append((img_path, classes_present.pop()))

print(f"{len(single_species_images)} of {len(list(test_img_dir.glob('*.jpg')))} test images contain exactly one species.")

# Evaluate on a manageable sample for speed.
EVAL_SAMPLE = 150
rng = np.random.default_rng(RNG_SEED)
eval_sample = [single_species_images[i] for i in rng.choice(len(single_species_images), size=min(EVAL_SAMPLE, len(single_species_images)), replace=False)]

prompts = [f"a photo of a {c}" for c in CLASSES]
text_tokens = clip.tokenize(prompts).to(DEVICE)
with torch.no_grad():
    text_features = F.normalize(clip_model.encode_text(text_tokens), dim=-1)

correct = 0
for img_path, true_cls in eval_sample:
    image_input = clip_preprocess(Image.open(img_path)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        image_features = F.normalize(clip_model.encode_image(image_input), dim=-1)
        sims = (image_features @ text_features.T).squeeze(0)
    pred_cls = int(sims.argmax().item())
    correct += int(pred_cls == true_cls)

acc = correct / len(eval_sample)
print(f"\nZero-shot CLIP accuracy on {len(eval_sample)} single-species test images: {acc:.4f}")
print(f"No training. No annotation. Just {len(CLASSES)} English prompts.")

Compare that number, with zero training, against Lab 7's trained YOLO26 (which needed 1,052 annotated training images) and Lab 9's zero-shot YOLOE detection results. CLIP's job here is easier than either — whole-image classification, not localisation — so a direct comparison isn't apples-to-apples, but it's a useful anchor for how far "no training at all" can get you on this dataset.

## 4. Building an image search engine

The same embedding space that powers zero-shot classification also powers **retrieval**: embed a library of images once, embed a free-text query, and rank the library by cosine similarity. This is not a toy — it's the same idea behind real product image search.

We embed a sample of the full dataset once (the expensive part, done exactly once), then any number of text queries against it are near-instant.

In [ ]:
LIBRARY_SIZE = 300
all_test_imgs = sorted(test_img_dir.glob("*.jpg"))
library_paths = [all_test_imgs[i] for i in rng.choice(len(all_test_imgs), size=min(LIBRARY_SIZE, len(all_test_imgs)), replace=False)]

library_embeddings = []
BATCH = 32
with torch.no_grad():
    for i in range(0, len(library_paths), BATCH):
        batch_paths = library_paths[i:i + BATCH]
        batch_tensor = torch.stack([clip_preprocess(Image.open(p)) for p in batch_paths]).to(DEVICE)
        feats = F.normalize(clip_model.encode_image(batch_tensor), dim=-1)
        library_embeddings.append(feats.cpu())
library_embeddings = torch.cat(library_embeddings, dim=0)
print(f"Embedded {len(library_paths)} images into a {library_embeddings.shape[1]}-dim space.")


def image_search(query: str, top_k: int = 6):
    text_tokens = clip.tokenize([query]).to(DEVICE)
    with torch.no_grad():
        query_embed = F.normalize(clip_model.encode_text(text_tokens), dim=-1).cpu()
    sims = (library_embeddings @ query_embed.T).squeeze(1)
    top_idx = torch.topk(sims, k=top_k).indices.tolist()

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, idx in zip(axes.flat, top_idx):
        ax.imshow(Image.open(library_paths[idx])); ax.axis("off")
        ax.set_title(f"sim={sims[idx]:.3f}", fontsize=10)
    for ax in axes.flat[len(top_idx):]:
        ax.set_visible(False)
    plt.suptitle(f'Query: "{query}"', y=1.0, fontsize=13)
    plt.tight_layout(); plt.show()


image_search("a herd of zebras")

In [ ]:
# Try a few more queries — including ones that describe something specific,
# not just a bare species name. CLIP was never told "close-up" or "herd" or
# "drinking water" belong to any of our classes; it learned what those words
# mean from its own 400-million-pair training set, entirely separately.
image_search("a close-up of an elephant's face")

## 5. Visualising the embedding space

Section 3's classification worked because images of the same species land near each other in CLIP's embedding space, and near their own species' text prompt. Let's see that structure directly by projecting the library embeddings down to 2D.

In [ ]:
from sklearn.decomposition import PCA

library_true_classes = []
for p in library_paths:
    lbl_path = DATA_ROOT / "labels" / "test" / f"{p.stem}.txt"
    classes_present = set(read_yolo_labels(lbl_path))
    library_true_classes.append(classes_present.pop() if len(classes_present) == 1 else -1)
library_true_classes = np.array(library_true_classes)

pca = PCA(n_components=2, random_state=RNG_SEED)
coords = pca.fit_transform(library_embeddings.numpy())

CLASS_COLOURS = ["#d6336c", "#f59f00", "#2b8a3e", "#1c7ed6"]
fig, ax = plt.subplots(figsize=(8, 7))
for cls_id, name in enumerate(CLASSES):
    mask = library_true_classes == cls_id
    ax.scatter(coords[mask, 0], coords[mask, 1], color=CLASS_COLOURS[cls_id], label=name, alpha=0.7, s=40)
mixed_mask = library_true_classes == -1
if mixed_mask.any():
    ax.scatter(coords[mixed_mask, 0], coords[mixed_mask, 1], color="#888888", label="multi-species", alpha=0.4, s=25)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("CLIP embedding space (PCA to 2D), coloured by TRUE species\n(colours were never used during embedding — only for this plot)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**What to look for.** Reasonably clean clusters, one per species, despite CLIP never having seen a single labelled example of any of our four classes during its own training, and despite this being an unsupervised 2D projection of a 512-dimensional space (some information is necessarily lost, so don't expect perfect separation). This is the same phenomenon Lab 9 exploited for zero-shot detection, viewed from the representation side rather than the detection side: **the structure needed to tell these species apart was already present in CLIP's embedding space before we ever showed it a single wildlife photo.**

## 6. Prompt wording, revisited

Lab 9 Exercise 1(a) found that prompt wording changes YOLOE's zero-shot detection accuracy. That sensitivity comes from exactly the CLIP text encoder you're using directly today — different phrasings land at different points in the embedding space, some closer to how the training data actually described these animals.

In [ ]:
def zeroshot_accuracy_with_prompts(prompt_template: str, sample=eval_sample):
    prompts = [prompt_template.format(c) for c in CLASSES]
    text_tokens = clip.tokenize(prompts).to(DEVICE)
    with torch.no_grad():
        text_features = F.normalize(clip_model.encode_text(text_tokens), dim=-1)
    correct = 0
    for img_path, true_cls in sample:
        image_input = clip_preprocess(Image.open(img_path)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_features = F.normalize(clip_model.encode_image(image_input), dim=-1)
            sims = (image_features @ text_features.T).squeeze(0)
        correct += int(int(sims.argmax().item()) == true_cls)
    return correct / len(sample)


templates = {
    "bare name":            "{}",
    "'a photo of a X'":     "a photo of a {}",
    "'a photo of a wild X'":"a photo of a wild {}",
    "'a X in the savanna'": "a {} in the savanna",
}
print(f"{'Prompt style':<26s}{'Accuracy':>10s}")
print("-" * 36)
for name, template in templates.items():
    acc = zeroshot_accuracy_with_prompts(template)
    print(f"{name:<26s}{acc:>10.4f}")

Richer, more caption-like phrasings ("a photo of a wild X in the savanna") often — though not always — outperform the bare species name, for the same reason Lab 9 found: CLIP's training data was captions, not class labels, so prompts that *read like a caption* tend to land closer to where the model actually learned to place that concept.

---

## 7. Exercise 1 — retrieval quality, quantified

Section 4's search results looked plausible by eye, but "looked plausible" is exactly the kind of claim Lab 8 taught you not to accept without measuring.

1. For each of the four species, run `image_search`-style retrieval (reuse the `image_search` function's similarity computation, but return indices instead of just plotting) with the query `"a photo of a {species}"`, `top_k=20`.
2. Using `library_true_classes`, compute **precision@20** for each species: of the top 20 results, what fraction actually contain that species (either as the sole species or among several)?
3. Report precision@20 for all four species. Is retrieval quality even across species, or does it track the same per-class pattern you'd expect from Section 3's classification results?

In [ ]:
# Your code for Exercise 1 here.

*Your precision@20 results per species, and what they show:*

## 8. Exercise 2 — CLIP vs your trained detector

Section 3 measured CLIP's zero-shot **classification** accuracy. Lab 8 measured your trained YOLO26's **detection** accuracy (mAP) on a genuinely independent held-out set. These aren't the same task, but a rough comparison is still informative.

1. On your Lab 8 held-out set (`../lab08_detection_evaluation/data/holdout_images` + `holdout_labels`), run CLIP zero-shot classification the same way as Section 3 (filtering to single-species images first).
2. Compare CLIP's accuracy on *your* held-out photos against its accuracy on the test split from Section 3. Is there a gap? If so, what does that tell you about how representative that split is of the images *you* went out and sourced?
3. Write 3-4 sentences comparing the effort each approach required (CLIP: zero, YOLO26: hours of annotation and GPU time) against the accuracy each achieved, framed as a build-vs-prompt decision in the spirit of Lab 9's framework.

In [ ]:
# Your code for Exercise 2 here.

*Your comparison:*

## 9. Exercise 3 — open-ended (pick one)

**Option A — Prompt ensembling.** CLIP's own zero-shot recipe (from the original paper) averages the text embeddings of *several* prompt templates per class, rather than using just one. Build 5-6 templates (e.g. `"a photo of a {}"`, `"a wild {}"`, `"a {} in its natural habitat"`, ...), embed all of them per class, average and re-normalise, then re-run Section 3's evaluation with these ensembled text embeddings. Does averaging help, hurt, or make no difference versus your best single prompt from Section 6?

**Option B — Find the duplicates.** Using the library embeddings from Section 4, find the most similar *pair* of distinct images (excluding an image compared to itself) by cosine similarity. Display them side by side. Are they near-duplicates (same photo session, same animal) or just visually similar? What would this technique be useful for in a real annotation pipeline (hint: think back to Lab 6's frame-leakage warning)?

**Option C — Where CLIP breaks.** Construct 5 deliberately hard or ambiguous test cases: a multi-species herd shot, a very distant/small animal, an unusual angle, a heavily cropped image, and one of your own photos from Lab 8's held-out set. Run zero-shot classification on each and report whether CLIP got it right. Write a paragraph on the failure pattern you see, if any, and how it compares to the failure patterns you saw from your trained YOLO26 in Lab 8's error analysis.

In [ ]:
# Your code for Exercise 3 (Option A, B, or C) here.

*Which option did you pick, and what did you find?*

---

## 10. Reflection questions

**Q1.** In your own words, explain what "contrastive" means in contrastive learning, and why a symmetric loss (image-to-text *and* text-to-image) is used rather than just one direction.

**Q2.** Lab 9's YOLOE and today's CLIP both perform zero-shot recognition, but they solve different problems (detection with boxes vs. whole-image classification/retrieval). What does YOLOE need beyond a plain CLIP text/image embedding comparison to produce a bounding box, and why can't CLIP alone do that?

**Q3.** Section 5's PCA plot showed species clustering in CLIP's embedding space despite CLIP never being trained on your four classes. Explain, referring back to Section 1, *why* this generalisation happens rather than treating it as a coincidence.

**Q4.** CLIP was trained on 400 million image-caption pairs scraped from the public internet, with no curation for balance or fairness. Name one concrete way this training data source could produce a biased or unreliable result for a task like ours (wildlife identification), and one way you might detect such a bias before trusting the model in a real deployment.

**Q5.** Referring to your findings across Sections 3, 4, and Exercise 2: for the specific case of a wildlife conservation charity choosing between CLIP zero-shot, YOLOE zero-shot (Lab 9), and a trained YOLO26 (Lab 7), what does each approach let you do that the others don't? There is no single right answer — justify your ranking with evidence from this lab and Lab 9.

*Your answers:*

**A1.**

**A2.**

**A3.**

**A4.**

**A5.**

---

## What's next

Every model in this module so far, including today's CLIP, has mapped **something to a representation** — an image to a class, a box, a mask, an explanation, an embedding. **Lab 14** asks the opposite question for the first time: can a model map a representation back to a **new, plausible image**? You'll build an autoencoder and put it to genuine use — detecting malaria-infected cells as anomalies, having only ever shown it healthy ones — then build a GAN and watch a network learn to generate images from nothing but random noise and an adversary trying to catch it out.

Before leaving today, make sure:

- [ ] You implemented the CLIP contrastive loss from first principles and saw it respond to matched vs random embeddings
- [ ] You ran zero-shot classification and recorded your accuracy
- [ ] You built and queried the image search engine with at least 3 different text queries
- [ ] You completed Exercises 1, 2, and 3
- [ ] You answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors**